# 02 — Inference (all models)

Runs all models sequentially on every pair in `lpips_eval_set.csv`.
Each model's outputs land in a separate folder so `03_evaluate.ipynb` can compare them.

**Models:**
- **A** — SD 1.5 img2img (structural baseline)
- **B** — InstructPix2Pix (instruction baseline)
- **C** — ControlNet + Canny (no fine-tuning)
- **D** — ControlNet + Canny + LoRA (fine-tuned on Penn campus)

Toggle which models to run with the boolean flags in the Config cell.
Each section clears GPU memory before loading the next model.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system diffusers==0.27.2 'transformers>=4.38.0,<5' 'huggingface-hub<0.26' accelerate controlnet-aux peft pillow-heif pandas opencv-python-headless
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# ── Config — edit these ──────────────────────────────────────────────────────
if IN_COLAB:
    BASE        = '/content/drive/My Drive/CIS_5190_group_project'
    LORA_DIR    = f'{BASE}/checkpoints/lora'   # set to None to skip model D
else:
    BASE        = '..'
    LORA_DIR    = f'{BASE}/checkpoints/lora'

EVAL_CSV    = f'{BASE}/lpips_eval_set.csv'
ALIGNED_DIR = f'{BASE}/aligned'

# Toggle which models to run
RUN_SD_BASELINE    = True
RUN_IP2P           = True
RUN_CONTROLNET     = True
RUN_CONTROLNET_LORA = True   # requires LORA_DIR to exist

# Inference hyperparams
SD_STRENGTH  = 0.55
GUIDANCE     = 7.5
NUM_STEPS    = 30
IP2P_IMG_GUIDANCE = 1.5
IP2P_STEPS   = 100
CANNY_LOW    = 100
CANNY_HIGH   = 200
COND_SCALE   = 1.0   # ControlNet conditioning scale

NEGATIVE = 'blurry, distorted, cartoon, painting, unrealistic, low quality'

In [ ]:
import gc, os
import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

eval_df = pd.read_csv(EVAL_CSV)
print(f'Eval set: {len(eval_df)} pairs')

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def load_anchor(row) -> Image.Image:
    return Image.open(os.path.join(ALIGNED_DIR, row['anchor_file'])).convert('RGB').resize((512, 512))

def prompt_for(row) -> str:
    loc = row['location'].replace('_', ' ').title()
    return (
        f'A photo of {loc} on the University of Pennsylvania campus '
        f'at {row["target_tod"].lower()}, {row["target_weather"].lower()} weather, '
        f'architectural photography, realistic lighting, high quality'
    )

def get_canny(img: Image.Image) -> Image.Image:
    arr = np.array(img.convert('L'))
    edges = cv2.Canny(arr, CANNY_LOW, CANNY_HIGH)
    return Image.fromarray(np.stack([edges]*3, axis=-1))

def run_inference(pipe_fn, out_dir: str, suffix: str):
    """pipe_fn(row, anchor_img) → PIL Image"""
    os.makedirs(out_dir, exist_ok=True)
    results = []
    for _, row in eval_df.iterrows():
        anchor = load_anchor(row)
        out_img = pipe_fn(row, anchor)
        stem = os.path.splitext(row['target_file'])[0]
        fname = f'{stem}{suffix}'
        out_img.save(os.path.join(out_dir, fname))
        results.append({**row.to_dict(), 'generated_file': fname})
    pd.DataFrame(results).to_csv(os.path.join(out_dir, 'results.csv'), index=False)
    print(f'  Saved {len(results)} images → {out_dir}')

## A — SD 1.5 img2img (baseline)

In [ ]:
if RUN_SD_BASELINE:
    from diffusers import AutoPipelineForImage2Image

    pipe_sd = AutoPipelineForImage2Image.from_pretrained(
        'stable-diffusion-v1-5/stable-diffusion-v1-5',
        torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
    )
    pipe_sd.enable_model_cpu_offload()

    def sd_fn(row, anchor):
        return pipe_sd(
            prompt=prompt_for(row), image=anchor,
            strength=SD_STRENGTH, guidance_scale=GUIDANCE,
            num_inference_steps=NUM_STEPS,
        ).images[0]

    print('Running SD baseline...')
    run_inference(sd_fn, f'{BASE}/outputs/sd_baseline', '_sd_baseline.jpg')

    del pipe_sd; free_gpu()
    print('GPU cleared.')

## B — InstructPix2Pix (baseline)

In [ ]:
if RUN_IP2P:
    from diffusers import StableDiffusionInstructPix2PixPipeline

    pipe_ip2p = StableDiffusionInstructPix2PixPipeline.from_pretrained(
        'timbrooks/instruct-pix2pix',
        torch_dtype=torch.float16, safety_checker=None,
    )
    pipe_ip2p.enable_model_cpu_offload()

    def ip2p_fn(row, anchor):
        instruction = f"Change this photo to {row['target_tod'].lower()} with {row['target_weather'].lower()} weather"
        return pipe_ip2p(
            prompt=instruction, image=anchor,
            num_inference_steps=IP2P_STEPS,
            image_guidance_scale=IP2P_IMG_GUIDANCE,
            guidance_scale=GUIDANCE,
        ).images[0]

    print('Running InstructPix2Pix...')
    run_inference(ip2p_fn, f'{BASE}/outputs/instructpix2pix', '_ip2p.jpg')

    del pipe_ip2p; free_gpu()
    print('GPU cleared.')

## C — ControlNet + Canny (no LoRA)

In [ ]:
if RUN_CONTROLNET:
    from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

    controlnet = ControlNetModel.from_pretrained(
        'lllyasviel/sd-controlnet-canny', torch_dtype=torch.float16
    )
    pipe_cn = StableDiffusionControlNetPipeline.from_pretrained(
        'stable-diffusion-v1-5/stable-diffusion-v1-5',
        controlnet=controlnet, torch_dtype=torch.float16, use_safetensors=True,
    )
    pipe_cn.scheduler = UniPCMultistepScheduler.from_config(pipe_cn.scheduler.config)
    pipe_cn.enable_model_cpu_offload()

    def cn_fn(row, anchor):
        return pipe_cn(
            prompt=prompt_for(row), negative_prompt=NEGATIVE,
            image=get_canny(anchor),
            num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE,
            controlnet_conditioning_scale=COND_SCALE,
        ).images[0]

    print('Running ControlNet (no LoRA)...')
    run_inference(cn_fn, f'{BASE}/outputs/controlnet', '_controlnet.jpg')

    del pipe_cn, controlnet; free_gpu()
    print('GPU cleared.')

## D — ControlNet + Canny + LoRA

In [ ]:
lora_ready = RUN_CONTROLNET_LORA and LORA_DIR and os.path.exists(LORA_DIR)
if RUN_CONTROLNET_LORA and not lora_ready:
    print(f'Skipping model D — LORA_DIR not found: {LORA_DIR}')

if lora_ready:
    from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

    controlnet = ControlNetModel.from_pretrained(
        'lllyasviel/sd-controlnet-canny', torch_dtype=torch.float16
    )
    pipe_cnl = StableDiffusionControlNetPipeline.from_pretrained(
        'stable-diffusion-v1-5/stable-diffusion-v1-5',
        controlnet=controlnet, torch_dtype=torch.float16, use_safetensors=True,
    )
    pipe_cnl.scheduler = UniPCMultistepScheduler.from_config(pipe_cnl.scheduler.config)
    pipe_cnl.load_lora_weights(LORA_DIR)
    pipe_cnl.enable_model_cpu_offload()
    print(f'LoRA loaded from {LORA_DIR}')

    def cnl_fn(row, anchor):
        return pipe_cnl(
            prompt=prompt_for(row), negative_prompt=NEGATIVE,
            image=get_canny(anchor),
            num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE,
            controlnet_conditioning_scale=COND_SCALE,
        ).images[0]

    print('Running ControlNet + LoRA...')
    run_inference(cnl_fn, f'{BASE}/outputs/controlnet_lora', '_controlnet_lora.jpg')

    del pipe_cnl, controlnet; free_gpu()
    print('GPU cleared.')

## Side-by-side preview
Shows one eval pair across all models that finished.

In [ ]:
import matplotlib.pyplot as plt

MODEL_OUTPUTS = {
    'SD Baseline':        (f'{BASE}/outputs/sd_baseline',      '_sd_baseline.jpg'),
    'InstructPix2Pix':    (f'{BASE}/outputs/instructpix2pix',  '_ip2p.jpg'),
    'ControlNet':         (f'{BASE}/outputs/controlnet',       '_controlnet.jpg'),
    'ControlNet + LoRA':  (f'{BASE}/outputs/controlnet_lora',  '_controlnet_lora.jpg'),
}

row = eval_df.iloc[0]
stem = os.path.splitext(row['target_file'])[0]
anchor = Image.open(os.path.join(ALIGNED_DIR, row['anchor_file']))
gt     = Image.open(os.path.join(ALIGNED_DIR, row['warped_path']))

panels = [('Anchor (source)', anchor), ('Ground Truth', gt)]
for name, (out_dir, suffix) in MODEL_OUTPUTS.items():
    p = os.path.join(out_dir, stem + suffix)
    if os.path.exists(p):
        panels.append((name, Image.open(p)))

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
for ax, (title, img) in zip(axes, panels):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.suptitle(f"{row['location']}  →  {row['target_tod']} / {row['target_weather']}", y=1.02)
plt.tight_layout()
plt.show()